In [21]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [23]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [24]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'CumulativeAssetCoveredLengthKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'PeakAboveSATCount',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'Bm1Density',
        'Bm2Density',
        'B0Share',
        'B1Share',
        'Bm1Share',
        'Bm2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare',
        'POR',
        'CurrentCompletion'
    ]


In [25]:
query = f"""DROP VIEW IF EXISTS Weekly_KPI;"""
cursor.execute(query)
conn.commit()


In [26]:
query = """
CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Week'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [27]:
print(query)


CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WH

In [28]:
query = "SELECT * FROM Weekly_KPI WHERE Year = 2026 AND BoundaryRegion IS NULL;"
df = pd.read_sql_query(query, conn)
conn.close()

In [29]:
Query(query = f"SELECT * FROM KPI_Data WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = 'Cadent') AND KPIId = 'POR'").execute(KPIHub_Conn)

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated


In [30]:
df

,Year,PeriodValue,CustomerName,BoundaryRegion,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,CumulativeAssetCoveredLengthKm,ServicePipeKm,...,Bm2Density,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare,POR,CurrentCompletion
0,2026,3,ITALGAS,None,577.70,549.53,577.70,549.53,None,0.0,...,0.22,16.69,0.81,66.21,16.28,59.59,24.06,16.35,None,None
1,2026,4,ITALGAS,None,1195.25,1146.54,1195.25,1146.54,None,0.0,...,0.32,14.36,1.29,66.09,18.27,61.99,25.95,12.06,None,None
2,2026,5,ITALGAS,None,4263.09,4058.37,4263.09,4058.37,None,0.0,...,0.31,12.84,0.41,67.67,19.08,61.86,22.84,15.30,None,None
3,2026,6,ITALGAS,None,364.26,326.87,364.26,326.87,None,0.0,...,0.15,18.03,1.09,67.76,13.11,60.95,30.10,8.96,None,None
4,2026,7,ITALGAS,None,1834.79,1745.05,1834.79,1745.05,None,0.0,...,0.32,11.41,0.36,68.15,20.08,68.12,21.56,10.33,None,None
5,2026,8,ITALGAS,None,2309.54,2192.25,2309.54,2192.25,None,0.0,...,0.29,12.63,0.54,66.77,20.06,60.20,25.16,14.64,None,None
6,2026,9,ITALGAS,None,5735.51,5460.49,5735.51,5460.49,None,0.0,...,0.22,14.55,0.59,67.60,17.26,61.26,25.58,13.16,None,None
7,2026,10,ITALGAS,None,483.89,453.82,483.89,453.82,None,0.0,...,0.39,9.62,0.27,66.40,23.71,60.66,32.41,6.94,None,None
8,2026,11,ITALGAS,None,1753.96,1637.64,1753.96,1637.64,None,0.0,...,0.26,12.92,0.21,68.35,18.52,52.19,30.48,17.33,None,None
9,2026,12,ITALGAS,None,3629.54,3445.13,3629.54,3445.13,None,0.0,...,0.26,13.71,0.97,67.01,18.31,56.16,27.55,16.29,None,None
